In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "workspace"
schema = "apex_retail"
gold_schema = "GOLD_tables"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")
print(f"Schema {catalog}.{gold_schema} ready")

Schema workspace.GOLD_tables ready


In [0]:
dim_customer = spark.table(f"{catalog}.{schema}.silver_customer").select(
    "customer_sk", "customer_id", "age", "gender", "income_bracket", "loyalty_program",
    "membership_years", "churned", "marital_status", "number_of_children",
    "education_level", "occupation", "customer_zip_code", "customer_city", "customer_state",
    "effective_start_date", "effective_end_date", "is_active"
)

dim_customer.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.dim_customer")

print("dim_customer rows:", spark.table(f"{catalog}.{gold_schema}.dim_customer").count())

dim_customer rows: 1050


In [0]:
dim_product = spark.table(f"{catalog}.{schema}.silver_product").select(
    "product_sk", "product_id", "product_name", "product_brand", "product_category",
    "product_rating", "product_review_count", "product_stock", "product_return_rate",
    "product_size", "product_weight", "product_color", "product_material",
    "product_manufacture_date", "product_expiry_date", "product_shelf_life", "unit_price"
)

dim_product.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.dim_product")

print("dim_product rows:", spark.table(f"{catalog}.{gold_schema}.dim_product").count())

dim_product rows: 1041


In [0]:
unknown_product_row = spark.createDataFrame([{
    "product_sk": -1, "product_id": "Unknown", "product_name": "Unknown",
    "product_brand": "Unknown", "product_category": "Unknown", "product_rating": 0.0,
    "product_review_count": 0, "product_stock": 0, "product_return_rate": 0.0,
    "product_size": "Unknown", "product_weight": 0.0, "product_color": "Unknown",
    "product_material": "Unknown", "product_manufacture_date": "Unknown",
    "product_expiry_date": "Unknown", "product_shelf_life": "Unknown", "unit_price": 0.0
}], schema=spark.table(f"{catalog}.{gold_schema}.dim_product").schema)

dim_product_final = spark.table(f"{catalog}.{gold_schema}.dim_product").unionByName(unknown_product_row)

dim_product_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.dim_product")

print("dim_product rows now:", spark.table(f"{catalog}.{gold_schema}.dim_product").count())

dim_product rows now: 1042


In [0]:
dim_promotion = (spark.table(f"{catalog}.{schema}.silver_sales")
    .select("promotion_type")
    .distinct()
    .withColumn("promotion_sk", F.row_number().over(Window.orderBy("promotion_type")).cast("long"))
    .select("promotion_sk", "promotion_type"))

(dim_promotion.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")   # <-- forces old promotion_id column to actually drop
    .saveAsTable(f"{catalog}.{gold_schema}.dim_promotion"))

spark.table(f"{catalog}.{gold_schema}.dim_promotion").printSchema()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


root
 |-- promotion_sk: long (nullable = true)
 |-- promotion_type: string (nullable = true)



In [0]:
from pyspark.sql.functions import to_date, col

# Filter out rows where transaction_date isn't a valid date before building calendar dimension
sales_dates = (spark.table(f"{catalog}.{schema}.silver_sales")
    .select("transaction_date")
    .filter(col("transaction_date") != "Unknown")
    .withColumn("full_date", to_date("transaction_date"))
    .filter(col("full_date").isNotNull())
    .select("full_date")
    .distinct())

print("Rows with a valid transaction_date:", sales_dates.count())

dim_date = (sales_dates
    .withColumn("date_sk", F.date_format("full_date", "yyyyMMdd").cast("long"))
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin([1, 7]))
    .select("date_sk", "full_date", "year", "month", "day", "day_of_week", "week_of_year", "is_weekend"))

dim_date.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold_schema}.dim_date")

print("dim_date rows:", spark.table(f"{catalog}.{gold_schema}.dim_date").count())
dim_date.orderBy("full_date").show(5)

Rows with a valid transaction_date: 996
dim_date rows: 996
+--------+----------+----+-----+---+-----------+------------+----------+
| date_sk| full_date|year|month|day|day_of_week|week_of_year|is_weekend|
+--------+----------+----+-----+---+-----------+------------+----------+
|20200101|2020-01-01|2020|    1|  1|  Wednesday|           1|     false|
|20200102|2020-01-02|2020|    1|  2|   Thursday|           1|     false|
|20200103|2020-01-03|2020|    1|  3|     Friday|           1|     false|
|20200104|2020-01-04|2020|    1|  4|   Saturday|           1|      true|
|20200109|2020-01-09|2020|    1|  9|   Thursday|           2|     false|
+--------+----------+----+-----+---+-----------+------------+----------+
only showing top 5 rows


In [0]:
unknown_date_row = spark.createDataFrame([{
    "date_sk": -1, "full_date": None, "year": 0, "month": 0, "day": 0,
    "day_of_week": "Unknown", "week_of_year": 0, "is_weekend": False
}], schema=spark.table(f"{catalog}.{gold_schema}.dim_date").schema)

dim_date_final = spark.table(f"{catalog}.{gold_schema}.dim_date").unionByName(unknown_date_row)

dim_date_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.dim_date")

print("dim_date rows now:", spark.table(f"{catalog}.{gold_schema}.dim_date").count())

dim_date rows now: 997


In [0]:
silver_sales = spark.table(f"{catalog}.{schema}.silver_sales")

fact_sales = (silver_sales.alias("s")
    .join(
        spark.table(f"{catalog}.{gold_schema}.dim_customer").filter("is_active = true").select("customer_sk", "customer_id").alias("c"),
        "customer_id", "left"
    )
    .join(
        spark.table(f"{catalog}.{gold_schema}.dim_product").select("product_sk", "product_id").alias("p"),
        "product_id", "left"
    )
    .withColumn("product_sk", F.coalesce(F.col("product_sk"), F.lit(-1)))
    .join(
        spark.table(f"{catalog}.{gold_schema}.dim_promotion").select("promotion_sk", "promotion_type").alias("pr"),
        "promotion_type", "left"
    )
    .withColumn("txn_date_parsed", F.when(F.col("transaction_date") != "Unknown", F.to_date("transaction_date")))
    .withColumn("date_sk_join", F.date_format("txn_date_parsed", "yyyyMMdd").cast("long"))
    .join(
        spark.table(f"{catalog}.{gold_schema}.dim_date").select("date_sk").alias("d"),
        F.col("date_sk_join") == F.col("date_sk"), "left"
    )
    .withColumn("date_sk", F.coalesce(F.col("date_sk"), F.lit(-1)))
    .select(
        F.col("s.sales_sk").alias("sales_sk"),
        F.col("s.transaction_id").alias("transaction_id"),
        "customer_sk", "product_sk", "promotion_sk", "date_sk",
        F.col("s.quantity").alias("quantity"),
        F.col("s.unit_price").alias("unit_price"),
        F.col("s.discount_applied").alias("discount_applied"),
        F.col("s.total_sales").alias("total_sales"),
        F.col("s.payment_method").alias("payment_method"),
        F.col("s.store_location").alias("store_location"),
        F.col("s.transaction_hour").alias("transaction_hour"),
        F.col("s.day_of_week").alias("day_of_week"),
        F.col("s.week_of_year").alias("week_of_year"),
        F.col("s.month_of_year").alias("month_of_year"),
        F.col("s.holiday_season").alias("holiday_season"),
        F.col("s.season").alias("season"),
        F.col("s.weekend").alias("weekend"),
        F.col("s.promotion_id").alias("promotion_id")
    ))

fact_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{gold_schema}.fact_sales")

print("fact_sales rows:", spark.table(f"{catalog}.{gold_schema}.fact_sales").count())
print("Rows with null customer_sk:", fact_sales.filter("customer_sk IS NULL").count())
print("Rows with null product_sk:", fact_sales.filter("product_sk IS NULL").count())
print("Rows with null date_sk:", fact_sales.filter("date_sk IS NULL").count())

fact_sales rows: 2000
Rows with null customer_sk: 0
Rows with null product_sk: 0
Rows with null date_sk: 0


In [0]:
# Compare product_id values/types on both sides
print("silver_sales product_id dtype:", dict(silver_sales.dtypes)["product_id"])
print("dim_product product_id dtype:", dict(spark.table(f'{catalog}.{gold_schema}.dim_product').dtypes)["product_id"])

silver_sales.select("product_id").show(5)
spark.table(f"{catalog}.{gold_schema}.dim_product").select("product_id").show(5)

# Check for a sample of unmatched product_ids
unmatched = (silver_sales.select("product_id").distinct()
    .join(spark.table(f"{catalog}.{gold_schema}.dim_product").select("product_id"), "product_id", "left_anti"))
unmatched.show(10)
print("Distinct unmatched product_ids:", unmatched.count())

In [0]:
total_distinct_sales_products = silver_sales.select("product_id").distinct().count()
total_dim_products = spark.table(f"{catalog}.{gold_schema}.dim_product").select("product_id").distinct().count()

print("Distinct product_ids referenced in sales:", total_distinct_sales_products)
print("Distinct product_ids in dim_product:", total_dim_products)
print("Unmatched (in sales but not in product master):", unmatched.count())